# Phase 1A: Monte Carlo Basin Stability

Dimension-independent basin stability (Menck et al. 2013) for PINN training.

**Key features:**
- Each run saved individually to JSONL → resilient to Colab disconnects
- On reconnect: re-run this cell, it skips completed seeds automatically
- FP64 precision (per FP64 paper, NeurIPS 2025)

**Before running:** Runtime → Change runtime type → GPU (A100 preferred)

In [ ]:
# Cell 1: Setup
import os
if not os.path.exists('math-ai'):
    !git clone https://github.com/dzmitrybudzko/math-ai.git
else:
    !cd math-ai && git pull

import sys
sys.path.insert(0, 'math-ai/experiments/phase1')

import torch
import numpy as np
import time

print(f'PyTorch {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'Memory: {mem_gb:.1f} GB')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'\nUsing device: {device}')

In [ ]:
# Cell 2: Verify single run timing + FP64
from convection_pinn import set_precision, train_single, CORRECT, TRIVIAL, DIVERGED
from mc_basin_stability import run_mc_stability, run_multi_beta, compute_basin_stability

set_precision(True)  # FP64
print(f'Default dtype: {torch.get_default_dtype()}')

# CUDA warmup
if device == 'cuda':
    _ = torch.randn(256, 256, device=device) @ torch.randn(256, 256, device=device)
    torch.cuda.synchronize()

# Time a single FP64 run
t0 = time.time()
outcome, l2, loss = train_single(seed=0, beta=30.0, method='adam', n_adam=5000, device=device)
dt = time.time() - t0

labels = {CORRECT: 'CORRECT', TRIVIAL: 'TRIVIAL', DIVERGED: 'DIVERGED'}
print(f'Single run (FP64, beta=30, 5000 Adam steps): {dt:.1f}s')
print(f'Outcome: {labels[outcome]}, L2={l2:.6f}')
print(f'\nEstimated times:')
for n in [100, 500, 1000]:
    print(f'  {n} runs -> ~{n*dt/60:.0f} min ({n*dt/3600:.1f} h)')

In [ ]:
# Cell 3: Quick verification (50 runs, beta=1)
# Should finish in ~5 min on A100
stats_verify = run_mc_stability(
    beta=1.0, method='adam', n_runs=50, n_adam=5000,
    lr=1e-3, device=device, fp64=True, out_dir='results_phase1a'
)
print(f'\nVerification passed: P(correct) = {stats_verify["p_correct"]:.3f}')

## Production Run

Each beta saves per-run to JSONL. If disconnected:
1. Reconnect to Colab
2. Re-run Cell 1 + Cell 2 (setup)
3. Re-run Cell 4 — it auto-skips completed runs

**Estimated time:** ~1000 runs × 5 betas × ~10s/run = ~14 hours on A100

In [ ]:
# Cell 4: Production — 1000 runs per beta
# Safe to re-run: skips completed seeds automatically

BETAS = [1.0, 5.0, 10.0, 30.0, 50.0]
N_RUNS = 1000
N_ADAM = 5000
LR = 1e-3

all_stats = run_multi_beta(
    betas=BETAS, method='adam', n_runs=N_RUNS, n_adam=N_ADAM,
    lr=LR, device=device, fp64=True, out_dir='results_phase1a'
)

In [ ]:
# Cell 5: Visualize basin stability vs beta
import matplotlib.pyplot as plt

betas = sorted(all_stats.keys())
p_correct = [all_stats[b]['p_correct'] for b in betas]
p_trivial = [all_stats[b]['p_trivial'] for b in betas]
p_diverged = [all_stats[b]['p_diverged'] for b in betas]
ci_low = [all_stats[b]['ci_correct_95'][0] for b in betas]
ci_high = [all_stats[b]['ci_correct_95'][1] for b in betas]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left: stacked bar chart
ax1.bar(range(len(betas)), p_correct, color='green', alpha=0.7, label='Correct')
ax1.bar(range(len(betas)), p_trivial, bottom=p_correct, color='orange', alpha=0.7, label='Trivial')
bottoms = [c + t for c, t in zip(p_correct, p_trivial)]
ax1.bar(range(len(betas)), p_diverged, bottom=bottoms, color='gray', alpha=0.7, label='Diverged')
ax1.set_xticks(range(len(betas)))
ax1.set_xticklabels([f'β={int(b)}' for b in betas])
ax1.set_ylabel('Fraction')
ax1.set_title('Basin Stability: Outcome Distribution vs β')
ax1.legend()
ax1.set_ylim(0, 1)

# Right: P(correct) with confidence intervals
yerr_low = [p - l for p, l in zip(p_correct, ci_low)]
yerr_high = [h - p for p, h in zip(p_correct, ci_high)]
ax2.errorbar(betas, p_correct, yerr=[yerr_low, yerr_high],
             fmt='o-', capsize=5, color='green', linewidth=2)
ax2.set_xlabel('β (PDE difficulty)')
ax2.set_ylabel('P(correct) — Basin Stability')
ax2.set_title('Basin Stability vs PDE Difficulty')
ax2.set_xscale('log')
ax2.grid(True, alpha=0.3)
ax2.set_ylim(0, 1)

plt.tight_layout()
plt.savefig('results_phase1a/basin_stability_vs_beta.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results_phase1a/basin_stability_vs_beta.png')

In [ ]:
# Cell 6: k-NN Basin Entropy (Phase 1B post-processing)
from knn_basin_entropy import analyze_mc_results
from mc_basin_stability import get_results_path

print('k-NN Basin Entropy analysis (dimension-independent)\n')

entropy_by_beta = {}
for beta in BETAS:
    jsonl_path = get_results_path('results_phase1a', beta, 'adam', N_ADAM, LR, True)
    if jsonl_path.exists():
        print(f'\n--- beta={int(beta)} ---')
        result = analyze_mc_results(
            str(jsonl_path), beta=beta, method='adam', n_adam=N_ADAM,
            lr=LR, fp64=True, device='cpu',
            k_values=[5, 10, 20, 50]
        )
        entropy_by_beta[beta] = result['entropy_results']

In [ ]:
# Cell 7: Download results
from google.colab import files
import glob as globmod

for f in sorted(globmod.glob('results_phase1a/*')):
    print(f'Downloading: {f}')
    files.download(f)